# STACKMATHQA - ALL ANSWERS EXTRACTION

This notebook links our 100 sampled questions to two datasets — **' stackmathqa200k '** and **' stackmathqafull-qalist '** — from which we recover per-question metadata and the full list of user-submitted answers available on the Stack Exchange network.

## Setup

In [2]:
from datasets import load_dataset
import sympy as sp
import re
import json
import os
import gc
from datetime import datetime
import random
import pandas as pd

In [1]:
from huggingface_hub import notebook_login

notebook_login()

## LOADING OUR QUESTIONS FROM EVALUATION

In [3]:
import json
import hashlib
from datasets import load_dataset
import pandas as pd

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

my_results = load_jsonl("results/eval_fullnew_20260603_161654_meta-llama/Meta-Llama-3.1-8B-Instruct.jsonl")

def hash_pair(q, a):
    text = (q.strip() + "\n" + a.strip()).encode("utf-8")
    return hashlib.md5(text).hexdigest()

lookup = {
    hash_pair(entry["question"], entry["correct_answer"]): entry
    for entry in my_results
}

print("Loaded results:", len(lookup))


Loaded results: 100


## RECOVER METADATA

In [4]:
stream = load_dataset(
    "math-ai/StackMathQA",
    "stackmathqa200k",
    split="train",
    streaming=True
)

recovered = []

for row in stream:
    h = hash_pair(row["Q"], row["A"])
    if h in lookup:
        merged = {
            **lookup[h],          # your model output
            "meta": row["meta"],  # metadata
            "original_answer": row["A"]
        }
        recovered.append(merged)

    if len(recovered) == len(lookup):
        break

print("Recovered:", len(recovered))


Recovered: 100


## FIND ALL ANSWERS OF THIS QUESTION

In [5]:
import hashlib

def hash_q(q):
    return hashlib.md5(q.strip().encode("utf-8")).hexdigest()

target_hashes = { hash_q(item["question"]) for item in recovered }

In [6]:
qalist_map = {}

stream_qalist = load_dataset(
    "math-ai/StackMathQA",
    "stackmathqafull-qalist",
    split="train",
    streaming=True
)

for row in stream_qalist:
    h = hash_q(row["Q"])
    if h in target_hashes:
        qalist_map[h] = row["A_list"]   # store the list of answers
    if len(qalist_map) == len(target_hashes):
        break

print("Found:", len(qalist_map))


Found: 100


In [8]:
for hash in qalist_map.keys():
    count = 0
    for item in recovered:
        if hash_q(item["question"]) == hash:
            count += 1
            item["A_list"] = qalist_map[hash]
            break
    if count > 1:
        print("Warning: multiple matches for hash", hash)

In [11]:
recovered_df = pd.DataFrame(recovered)
recovered_df["num_answers"] = recovered_df["A_list"].apply(len)
print(recovered_df["num_answers"].max())

7


In [7]:
# Map each of your 100 questions to its canonical hash
canonical_map = {}

for item in recovered:
    q = item["question"].strip()
    h = hash_q(q)
    canonical_map.setdefault(h, []).append(q)

# Now detect duplicates
duplicates = {h: qs for h, qs in canonical_map.items() if len(qs) > 1}

print("Number of duplicate canonical questions:", len(duplicates))

if duplicates:
    print("\nDuplicates found:")
    for h, qs in duplicates.items():
        print("----")
        for q in qs:
            print(q[:200], "...")


Number of duplicate canonical questions: 0


In [14]:
recovered_df.to_json("results/merged_results_from_llama.jsonl", orient="records", lines=True, force_ascii=False)
recovered_df.to_csv("results/merged_results_from_llama.csv", index=False)